
## Predictive NPS Case
Predict customer satisfaction (NPS) before the survey is deployed.

# Phase 1 - Business Understanding

## 1. Business context

A national e-commerce company handles a growing volume of orders, deliveries
and customer interactions. That scale brought real efficiency gains,
but it also surfaced important challenges in customer experience, especially
the high variability of Net Promoter Score (NPS) across different customer
segments. Customer Experience noticed that, even when operational
metrics look similar, some customers become brand promoters while others become detractors.

Today, NPS is collected **only after the purchase journey is complete**, which limits the company's ability to:
- Anticipate issues before they shape customer perception
- Prioritize corrective actions in logistics and customer service
- Act **proactively**, not only reactively

NPS measures how likely a customer is to recommend the company to others, on a 0–10 scale:

| Category | Range | Behavior |
|----------|-------|----------|
| **Promoters** | 9 – 10 | Loyal customers who actively recommend the brand |
| **Passives** | 7 – 8 | Satisfied but not actively engaged |
| **Detractors** | 0 – 6 | Dissatisfied; may harm brand reputation |

In this context, the company needs to turn operational data—orders, logistics, and
service interactions—into actionable insights that support strategic and
operational decisions.

### Why is NPS strategic for e-commerce?

#### Impact on repeat purchase
Promoters are significantly more likely to buy again. In e-commerce, where customer acquisition cost (CAC) is high, retaining an existing customer is far more profitable than winning a new one. Customers with high NPS tend to have higher lifetime value (LTV) and lower churn, which directly affects recurring revenue.

#### Impact on word of mouth
Online, recommendations (or criticism) spread fast: marketplace ratings, social posts, and review platforms shape how new buyers perceive the brand. A single vocal detractor can sway many purchase decisions. An engaged promoter, in turn, acts as zero-cost organic marketing.

#### Impact on market share
In a competitive e-commerce market, customer experience is a key differentiator. Companies with consistently high NPS tend to outgrow the market by combining lower retention cost with higher organic acquisition. A drop in NPS is an early signal of market share loss often before revenue numbers turn down.


## 2. Business objectives

### Objective 1 — Diagnostics

The company wants to understand which operational factors drive dissatisfaction, which variables are linked to it, and which influence dissatisfaction the most. Success is measured by identifying the top 3 variables with the strongest correlation with NPS ≤ 6.

### Objective 2 — Prevention

Reduce detractors by 10% within 90 days with the model in production, using automated preventive actions for customers flagged as likely future detractors before the NPS survey.


## 3. Analytical objectives

Analytical objectives translate business goals into technical data science problems.

### Analytical objective 1 — Diagnostic analysis

> **Run a diagnostic and exploratory analysis (EDA) to identify which operational variables—logistics (`delivery_delay_days`, `delivery_attempts`), service (`customer_service_contacts`, `resolution_time_days`, `complaints_count`), and order-level (`order_value`, `freight_value`)—show the strongest association with NPS ≤ 6 (Detractor), using descriptive statistics, visualizations, and correlation analysis.**

- **Problem type:** Descriptive (why did it happen?)
- **Techniques:** EDA, correlation, segmentation by NPS groups

### Analytical objective 2 — Predictive model

> **Build a binary classification model that predicts whether an order (unit of analysis) will receive NPS ≤ 6 (Detractor) after the purchase journey ends and before the survey is fielded, using only operational variables available at prediction time.**

- **Problem type:** Predictive (binary classification)
- **Target variable:** `nps_detrator` → 1 if `nps_score ≤ 6`, 0 otherwise
- **Unit of analysis:** Order (`order_id`)
- **Horizon:** After delivery and before the NPS survey
- **Evaluation metrics:** Recall ≥ 75% (priority: catch detractors), AUC-ROC ≥ 0.80, F1-score as a balance metric


#### Why is recall the priority metric?
For customer satisfaction, a **false negative** (missing a future detractor) is costlier than a **false positive** (triggering prevention for a customer who would not be a detractor). Offering a coupon to a satisfied customer is cheap; leaving a dissatisfied customer without action is costly in reputation and revenue.

#### Variable excluded from the model
`csat_internal_score` **must not be used as a feature in the predictive model.**  
**Leakage** risk: this internal satisfaction score may be computed from information collected at the same time as or after NPS, behaving like a future variable the model would not have at real prediction time. Using it would artificially inflate training performance.


## 4. Success criteria

Success criteria define how we will know the project worked—both technically and for the business. They are the bar used in the Evaluation phase.

### Technical criteria (model metrics)

| Metric | Minimum target | Rationale |
|--------|----------------|-----------|
| **Recall** (detractors) | ≥ 75% | Priority is to catch who will rate poorly; false negatives are costlier |
| **AUC-ROC** | ≥ 0.80 | Overall ability to separate detractors from non-detractors |
| **F1-score** | ≥ 0.70 | Balance between precision and recall |
| **Precision** (detractors) | ≥ 60% | Avoid too many false positives that overload CRM |

> **Note:** Accuracy is **not** the primary metric. If roughly 25% of the base are detractors, a trivial model (never predicting detractor) would show 75% accuracy—with no real value.

### Business criteria (real impact)

| Criterion | Target | How to measure |
|-----------|--------|----------------|
| Detractor reduction | Reduce detractor share by ≥ 10% within 90 days | Before/after comparison with the model in production |
| Preventive action effectiveness | ≥ 30% of customers flagged by the model avoid NPS ≤ 6 | A/B test: preventive action vs. control |
| Response time | Score produced within 24 hours of delivery | Operational pipeline monitoring |


## 5. Target definition

### Which variable represents customer satisfaction?
The variable `nps_score` represents customer satisfaction. It is a 0–10 score provided by the customer after the purchase journey, answering the classic NPS question: "On a scale of 0 to 10, how likely are you to recommend our company to a friend or colleague?"

### Why was it chosen?
It is the only variable in the dataset generated by the customer, not by the company. All other fields are internal operational records. NPS is the direct voice of the customer about their experience, making it the most legitimate satisfaction signal.

It is observable, measurable, and has a clear time horizon—the three criteria CRISP-DM expects from a good target. We know exactly who responded, what score they gave, and when.

`nps_score` ties directly to the business goal of reducing detractors. Using `nps_score` as the basis for the target ensures the model answers the business question without approximation.

### When in the journey is this information collected?
Customer journey: order placed → payment → picking → shipment → delivery → NPS survey

`nps_score` is collected at the last step, after the full purchase experience. That has a critical modeling implication: features must be limited to variables available before the survey is answered.

Variables available before NPS and valid as features: customer_id, order_id, customer_age, customer_region, customer_tenure_months, order_value, items_quantity, discount_value, payment_installments, delivery_time_days, delivery_delay_days, freight_value, delivery_attempts, customer_service_contacts, resolution_time_days, complaints_count.
Variables that need special attention:

**csat_internal_score**: internal satisfaction score produced by operations, likely computed from data that includes or is contemporaneous with NPS. It should be excluded from the model due to leakage risk—using it would give the model access to the future.

**repeat_purchase_30d**: indicates a repeat purchase within 30 days of the order. This needs investigation: if the repeat purchase happens before the NPS survey is answered, it may be a valid feature; if after, it is leakage. The timeline must be verified before use.


### Is there a risk of misusing this variable?

**Leakage:** The most serious risk in this dataset. `csat_internal_score` is a strong leakage candidate: if it is built from information that exists only after or at the same time as NPS, including it as a feature lets the model "see the future" during training. Performance would look artificially strong in the notebook and collapse in production.<br>
**Moving target definition:** Detractor is defined today as `nps_score ≤ 6`. If the company changes that threshold (for example to ≤ 5), a model trained on the old definition will be misaligned and must be rebuilt. Target definitions should be versioned and documented.<br>
**Target drift:** Even with a fixed definition, the share of detractors can shift over time due to logistics crises, seasonality, or customer mix changes. A model trained in a low-churn period may degrade in a crisis period, so production models need continuous monitoring.



*The variable `nps_detrator` will be created in the Data Preparation phase as:<br>
    nps_detrator = 1 if nps_score ≤ 6, otherwise 0.*
